(a) How much peak memory does running AdamW require? Decompose your answer based on the 
memory usage of the parameters, activations, gradients, and optimizer state. Express your 
answer in terms of the batch_size and the model hyperparameters (vocab_size, 
context_length, num_layers, d_model, num_heads). Assume 𝑑ff = 8/3 × 𝑑_model.
For simplicity, when calculating memory usage of activations, consider only the following 
components:
• Transformer block
    ‣RMSNorm(s)
    ‣Multi-head self-attention sublayer: 𝑄𝐾𝑉  projections, 𝑄𝐾⊤ matrix multiply, softmax, weighted sum of values, output projection.
    ‣Position-wise feed-forward (SwiGLU): 𝑊1, 𝑊2, SiLU on the gate branch, element-wise product, 𝑊3
• final RMSNorm
• output embedding
• cross-entropy on logits
Deliverable: An algebraic expression for each of parameters, activations, gradients, and 
optimizer state, as well as the total.


B=batch_size
T=context_length
L=num_layers
D=d_model
H=num_heads
V=vocab_size
d_ff=F=8/3D

Total Parameters(P) = 2VD + L(4DD + 8DD + 2D) + D
Total Gradients(G) = P
Optimizer State(S) = 2P
Activactions(A) = L[2BTD + (3BTD + BHTT + BHTT + BTD + BTD) + (2BTF + BTF + BTF + BTD)] + BTD + BTV + BTV

Memory Usage = P + G + S + A = 4P + A

Peak Memory = Memory Usage * size(float32)

(b) Instantiate your answer for a GPT-2 XL-shaped model to get an expression that only 
depends on the batch_size. What is the maximum batch size you can use and still fit within 
80GB memory?
Deliverable: An expression that looks like 𝑎 ⋅ batch_size + 𝑏 for numerical values 𝑎, 𝑏, and a 
number representing the maximum batch size.

In [32]:
B=batch_size=3.65
T=context_length=1024
L=num_layers=48
D=d_model=1600
H=num_heads=25
V=vocab_size=50257
F=4288

P = 2*V*D + L*(12*D*D + 2*D) + D
A = L*(8*B*T*D + 2*B*H*T*T  + 4*B*T*F) + B*T*D + 2*B*T*V
A_noL = (8*B*T*D + 2*B*H*T*T  + 4*B*T*F) + B*T*D + 2*B*T*V
memeory_usage = (4 * P + A) * 4

GB = 1024 * 1024 * 1024

print("memeory_usage: ", memeory_usage // GB, " GB")

memeory_usage:  80.0  GB


最大 batch size = 3

(c) How many FLOPs does running one step of AdamW take?

B=batch_size
T=context_length
L=num_layers
D=d_model
H=num_heads
V=vocab_size
d_ff=F=8/3D

P = 2*V*D + L*(12*D*D + 2*D) + D

AdamW 的一步（one optimizer step）对每个参数需要执行:
1. Weight decay: 2P
2. First moment update: 3P
3. Second moment update: 4P
4. Parameter update; 5P

FLOPs = 14P

(d) Model FLOPs utilization (MFU) is defined as the ratio of observed throughput (tokens per 
second) relative to the hardware’s theoretical peak FLOP throughput. An NVIDIA H100 GPU has a theoretical peak of 495 teraFLOPs 
for “float32” (actually TensorFloat-32, which in reality is “bfloat19”) operations. Assuming 
you are able to get 50% MFU, how long would it take to train a GPT-2 XL for 400K steps 
and a batch size of 1024 on a single H100? assume that the backward pass has twice the FLOPs of the forward pass.
Deliverable: The number of hours training would take, with a brief justification.

In [ ]:
B=batch_size=1024
T=context_length=1024
L=num_layers=48
D=d_model=1600
H=num_heads=25
V=vocab_size=50257
F=4288

step = 400 * 1e3

P = 2*V*D + L*(12*D*D + 2*D) + D

F_forward = 2 * P * B * T       # B * T 就是token数, 每个token大约需要 2P FLOPs
F_backward = 2 * F_forward      # assume that the backward pass has twice the FLOPs of the forward pass
F_total = (F_forward + F_backward) * step

h100_throughput = 4.95 * 1e14 * 0.5

time = F_total / h100_throughput
print("Traning time: ", time // 3600, " h")

Traning time:  4619.0  h
